# Prepare MMIDAS input data (snRNA / Dbh)

Reads the batch-normalized snRNA AnnData, extracts the `BN` (batch-normalized) layer, and writes the MMIDAS input `data/snRNA_BN_norm1.h5ad` — consumed by the augmenter/model training scripts and by `00b`.

**Data availability:** the source file `data/snRNAseq_LCNE_BN_d4_1-5k.h5ad` is not stored in git. Download it from Code Ocean and place it under `data/`:
https://codeocean.allenneuraldynamics.org/data-assets/915f30e7-324d-4ed5-a520-5509a6e54f87/LCNE-transcriptomics-preprocessing_2026-07-08_16-23-00

Downstream artifacts (augmenter `.pth`, trained models, result `.pkl`) are regenerated by the pipeline — see `00b`.

In [ ]:
import scanpy as sc
import anndata

# --- paths (relative to repo root; see Data availability note) ---
INPUT_H5AD = "data/snRNAseq_LCNE_BN_d4_1-5k.h5ad"   # source snRNA data with a batch-normalized ("BN") layer
OUTPUT_H5AD = "data/snRNA_BN_norm1.h5ad"            # MMIDAS input produced by this notebook

## 1. snRNA

In [ ]:
adata_foo = sc.read_h5ad(INPUT_H5AD)
# collapse per-cell concatenated sex labels (e.g. "M;M;...;M") to a single value
adata_foo.obs['sex'] = adata_foo.obs['sex'].str.split(';').str[0]

# keep only the batch-normalized ("BN") layer as the working matrix
adata_BN = anndata.AnnData(adata_foo.layers['BN'])
adata_BN.obs = adata_foo.obs.copy()
adata_BN.obsm = adata_foo.obsm.copy()
adata_BN.var = adata_foo.var.copy()
adata_sc = adata_BN.copy()

# quick sanity check: PCA of the BN matrix
sc.tl.pca(adata_sc)
a = sc.pl.pca(adata_sc, show=False)
a.set_aspect('equal')

In [ ]:
adata_sc.write_h5ad(OUTPUT_H5AD)